# Figure 5 — Trace-element maps of representative clinopyroxene crystals

**Caracciolo et al. — main-text figure 5**

Builds a 3 × 3 panel figure of **Cr, Sc and Zr** concentration maps (rows) for one representative clinopyroxene crystal from each eruption (columns): **El Charco 1712**, **Teneguía 1971** and **Tajogaite 2021**.

Each panel shows the log10 concentration with a **fixed per-element colour scale**, so maps of the same element are directly comparable between eruptions; the colourbar label additionally reports the actual 1–99 percentile concentration range of that crystal.

The notebook is fully self-contained: it only needs the Python packages listed in the accompanying README and the Zenodo dataset folder `Maps_SingleSpots_LaPalma` placed next to this notebook.

## 1. Imports

In [ ]:
import string
from pathlib import Path

import numpy as np
import numpy.ma as ma
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Matplotlib export settings: editable text in PDF/SVG, Arial font
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"

## 2. Data configuration

The data location, the element channels and the metadata of all available crystal maps (folder path, image resolution, scale-bar position).

In [ ]:
# keep the original folder names and directory structure
DATA_ROOT  = Path("./Maps_SingleSpots_LaPalma")   # dataset root
MAP_SUBDIR = "time00000_level00000"               # subfolder with the channel CSVs
FIGURE_DIR = Path("./Figures")                    # all output figures go here


# Channels: CSV file name (without ".csv") for each measured element
CHANNEL_FILES = {
    "Ca": "channel_Ca _42Ca_",
    "Cr": "channel_Cr _52Cr_",
    "Sc": "channel_Sc _45Sc_",
    "Zr": "channel_Zr _90Zr_",
    "Mn": "channel_Mn _55Mn_",
    "Ni": "channel_Ni _60Ni_",
    "Sr": "channel_Sr _88Sr_",
    "V":  "channel_V _51V_",
    "Ti": "channel_Ti _47Ti_",
    "Ce": "channel_Ce _140Ce_",
}

# The Ca channel is used only for masking: pixels with Ca counts below
# These thresholds are background (epoxy/holes) and are excluded.
MASK_ELEMENT = "Ca"
CA_BACKGROUND_THRESHOLD = 1e5


# Samples: one entry per crystal map
SAMPLES = [
    # --- El Charco 1712 ---
    dict(eruption="El_Charco_1712", title="El Charco 1712 – Cpx 12",
         path="ElCharco1712/Maps_1712/export_calibrated 1_63660_cpx12",
         scale_um_per_px=4.2, bar_loc="bottom_right"),
    dict(eruption="El_Charco_1712", title="El Charco 1712 – Cpx 13",
         path="ElCharco1712/Maps_1712/export_calibrated 1_63663_cpx13",
         scale_um_per_px=3, bar_loc="bottom_right"),
    dict(eruption="El_Charco_1712", title="El Charco 1712 – Cpx 1-2",
         path="ElCharco1712/Maps_1712/export_calibrated 1_63664B-cpx1-2",
         scale_um_per_px=3, bar_loc="bottom_right"),
    dict(eruption="El_Charco_1712", title="El Charco 1712 – Cpx 21",
         path="ElCharco1712/Maps_1712/export_calibrated 1_63671_cpx21",
         scale_um_per_px=4, bar_loc="bottom_right"),
    dict(eruption="El_Charco_1712", title="El Charco 1712 – Cpx 22",
         path="ElCharco1712/Maps_1712/export_calibrated 1_63671_cpx22",
         scale_um_per_px=5, bar_loc="bottom_right"),
    # --- Teneguia 1971 ---
    dict(eruption="Teneguia_1971", title="Teneguía 1971 – Cpx 4",
         path="Teneguia1971/Maps_1971/export_calibrated 1_43440_cpx4",
         scale_um_per_px=3, bar_loc="bottom_left"),
    dict(eruption="Teneguia_1971", title="Teneguía 1971 – Cpx 3-5",
         path="Teneguia1971/Maps_1971/export_calibrated 1_44136_cpx3-5",
         scale_um_per_px=3, bar_loc="bottom_left"),
    dict(eruption="Teneguia_1971", title="Teneguía 1971 – Cpx 1",
         path="Teneguia1971/Maps_1971/export_calibrated 1_44149_cpx1",
         scale_um_per_px=3, bar_loc="bottom_left"),
    dict(eruption="Teneguia_1971", title="Teneguía 1971 – Cpx 3",
         path="Teneguia1971/Maps_1971/export_calibrated 1_44149_cpx3",
         scale_um_per_px=3, bar_loc="bottom_left"),
    dict(eruption="Teneguia_1971", title="Teneguía 1971 – Cpx 2-5",
         path="Teneguia1971/Maps_1971/export_calibrated 1_44154_cpx2-5",
         scale_um_per_px=3, bar_loc="bottom_right"),
    # --- Tajogaite 2021 ---
    dict(eruption="Tajogaite_2021", title="Tajogaite 2021 – Cpx 3",
         path="2021LaPalma/Maps/118424-cpx3/export_calibrated 1_118424-cpx3",
         scale_um_per_px=3, bar_loc="bottom_right"),
    dict(eruption="Tajogaite_2021", title="Tajogaite 2021 – Cpx 2",
         path="2021LaPalma/Maps/118465_cpx2/export_calibrated 1_118465-cpx2",
         scale_um_per_px=3, bar_loc="bottom_right"),
    dict(eruption="Tajogaite_2021", title="Tajogaite 2021 – Cpx 0",
         path="2021LaPalma/Maps/118465_Test cpx/export_calibrated 1_Area",
         scale_um_per_px=3, bar_loc="bottom_left"),
    dict(eruption="Tajogaite_2021", title="Tajogaite 2021 – Cpx 4",
         path="2021LaPalma/Maps/118508_cpx4/export_calibrated 1_118508-cpx4_Map",
         scale_um_per_px=5, bar_loc="bottom_left"),
    dict(eruption="Tajogaite_2021", title="Tajogaite 2021 – Cpx 5",
         path="2021LaPalma/Maps/118537_cpx5/export_calibrated 1_118537-cpx5",
         scale_um_per_px=3, bar_loc="bottom_left"),
]

ERUPTION_DISPLAY_NAMES = {
    "El_Charco_1712": "El Charco 1712",
    "Teneguia_1971":  "Teneguía 1971",
    "Tajogaite_2021": "Tajogaite 2021",
}

print(f"Data root: {DATA_ROOT.resolve()}")
print(f"Available crystal maps: {len(SAMPLES)}")

## 3. Helper functions

In [ ]:
def channel_csv_path(sample, element):
    """Full path to the CSV file of one element channel of one sample."""
    return DATA_ROOT / sample["path"] / MAP_SUBDIR / f"{CHANNEL_FILES[element]}.csv"


def load_cube(sample, elements):
    """Load the requested element channels of one sample.

    Returns an array of shape (rows, cols, len(elements)), in the order
    given by 'elements'. Raises if a CSV file is missing or if the
    channel images do not all share the same spatial dimensions.
    """
    loaded = []
    for el in elements:
        fp = channel_csv_path(sample, el)
        if not fp.is_file():
            raise FileNotFoundError(
                f"Channel file not found: {fp}\n"
                f"Check that DATA_ROOT ({DATA_ROOT.resolve()}) points to the "
                "Zenodo dataset and that the folder structure was not modified.")
        loaded.append(pd.read_csv(fp, header=None).values.astype(float))

    first_shape = loaded[0].shape
    for el, arr in zip(elements, loaded):
        if arr.shape != first_shape:
            raise ValueError(
                f"Channel shape mismatch in sample {sample['path']!r}: "
                f"{el} has shape {arr.shape}, expected {first_shape}.")

    return np.stack(loaded, axis=-1)


def background_mask(ca_channel, threshold=CA_BACKGROUND_THRESHOLD):
    """Boolean mask of background pixels (True = background / masked)."""
    return ca_channel < threshold


def save_figure(fig, name, formats=("pdf",), dpi=300, transparent=True):
    """Save a figure into FIGURE_DIR in the requested formats."""
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    for ext in formats:
        out = FIGURE_DIR / f"{name}.{ext}"
        fig.savefig(out, bbox_inches="tight", format=ext,
                    dpi=dpi, transparent=transparent)
        print(f"Saved: {out}")


def sample_by_title(title):
    """Look up a crystal map in SAMPLES by its figure title."""
    for s in SAMPLES:
        if s["title"] == title:
            return s
    raise KeyError(f"No sample with title {title!r}. "
                   f"Available: {[s['title'] for s in SAMPLES]}")


def add_scale_bar(ax, image_shape, scale_um_per_px, bar_loc="bottom_right",
                  bar_length_um=100, bar_height_px=5, margin_px=10,
                  fontsize=11):
    """Draw a scale bar of 'bar_length_um' micrometres on a map axis."""
    bar_len_px = int(bar_length_um / scale_um_per_px)
    h, w = image_shape

    positions = {
        "bottom_right": (w - bar_len_px - margin_px, h - bar_height_px - margin_px),
        "bottom_left":  (margin_px, h - bar_height_px - margin_px),
        "top_left":     (margin_px, margin_px),
        "top_right":    (w - bar_len_px - margin_px, margin_px),
    }
    if bar_loc not in positions:
        raise ValueError(f"Unknown bar_loc {bar_loc!r}; "
                         f"choose one of {sorted(positions)}")
    x0, y0 = positions[bar_loc]

    ax.add_patch(Rectangle((x0, y0), bar_len_px, bar_height_px, color="black"))
    ax.text(x0 + bar_len_px / 2, y0 - 5, f"{bar_length_um} μm",
            ha="center", va="bottom", fontsize=fontsize)

## 4. Figure configuration

Choose **which elements** to plot (rows), **which crystal map** represents each eruption (columns), and the **fixed colour-scale ranges** in ppm.

To plot a different crystal, replace the title passed to `sample_by_title()` — any `title` listed in `SAMPLES` (section 2) works.

In [ ]:
# Elements shown as figure rows
PANEL_ELEMENTS = ["Cr", "Sc", "Zr"]

# Crystal map shown for each eruption (figure columns)
SELECTED_MAPS = {
    "El_Charco_1712": sample_by_title("El Charco 1712 – Cpx 22"),
    "Teneguia_1971":  sample_by_title("Teneguía 1971 – Cpx 2-5"),
    "Tajogaite_2021": sample_by_title("Tajogaite 2021 – Cpx 5"),
}

# Fixed colour-scale ranges (ppm) per element, shared across all eruptions
PPM_RANGES = {
    "Cr": (0.1, 1900),
    "Sc": (20, 140),
    "Zr": (80, 700),
}

SAVE_FIG    = True                              # set False to skip saving
OUTPUT_NAME = "MainTextFigure_fixed_logscale"   # saved as .png + .pdf in ./Figures

BAR_LENGTH_UM = 100                             # scale-bar length in micrometres

# Centralised font sizes
FONT = {
    "panel_label":    16,
    "col_title":      18,
    "row_label":      16,
    "scalebar":       11,
    "colorbar":       15,
    "colorbar_ticks": 12,
}

## 5. Build the figure

For each (element, eruption) panel:

1. Load only the two channels needed: Ca (background mask) + the element;
2. Mask background pixels (Ca below `CA_BACKGROUND_THRESHOLD`);
3. Compute the crystal's actual 1–99 percentile concentration range (reported on the colourbar label);
4. Plot `log10(ppm)` with the fixed per-element colour limits ("magma" colormap, background masked out);
5. Add the panel letter.

Column titles (eruptions) and row labels (elements) are added at the end, and the figure is saved as PNG (300 dpi) and PDF.

In [ ]:
panel_labels = list(string.ascii_uppercase)
eruptions = list(SELECTED_MAPS.keys())

# log10 colour limits from the fixed ppm ranges
log_ranges = {el: (np.log10(max(lo, 1e-3)), np.log10(hi))
              for el, (lo, hi) in PPM_RANGES.items()}

fig, axes = plt.subplots(nrows=len(PANEL_ELEMENTS), ncols=len(eruptions),
                         figsize=(15, 15), constrained_layout=True)

for i, element in enumerate(PANEL_ELEMENTS):
    vmin_log, vmax_log = log_ranges[element]

    for j, eruption in enumerate(eruptions):
        ax = axes[i, j]
        sample = SELECTED_MAPS[eruption]

        # Panel letter (A, B, C, ...)
        ax.text(0.98, 0.98, panel_labels[i * len(eruptions) + j],
                transform=ax.transAxes, fontsize=FONT["panel_label"],
                fontweight="bold", ha="right", va="top")

        # Load only the channels needed: Ca (mask) + element 
        cube = load_cube(sample, [MASK_ELEMENT, element])
        mask = background_mask(cube[:, :, 0])
        raw_ppm = cube[:, :, 1]

        # Actual 1-99 percentile range of the crystal (for the colourbar label;
        # the colour scale itself is fixed per element)
        valid = raw_ppm[~mask]
        vmin_ppm, vmax_ppm = np.percentile(valid, 1), np.percentile(valid, 99)

        # log10 display (clipped to avoid log(0)), background masked
        log_ppm = np.log10(np.clip(raw_ppm, 1e-3, None))
        data_masked = ma.array(log_ppm, mask=mask)

        im = ax.imshow(data_masked, cmap="magma", interpolation="nearest",
                       vmin=vmin_log, vmax=vmax_log)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)

        # Colourbar with the crystal's real concentration range
        cbar = fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
        cbar.set_label(f"{element} | [{vmin_ppm:.0f} – {vmax_ppm:.0f} ppm]",
                       fontsize=FONT["colorbar"])
        cbar.ax.tick_params(labelsize=FONT["colorbar_ticks"])

        add_scale_bar(ax, raw_ppm.shape, sample["scale_um_per_px"],
                      bar_loc=sample["bar_loc"],
                      bar_length_um=BAR_LENGTH_UM,
                      fontsize=FONT["scalebar"])

# Column titles (eruptions) and row labels (elements)
for j, eruption in enumerate(eruptions):
    fig.text((j + 0.42) / len(eruptions), 1.01,
             ERUPTION_DISPLAY_NAMES[eruption],
             ha="center", va="bottom",
             fontsize=FONT["col_title"], fontweight="bold")

for i, element in enumerate(PANEL_ELEMENTS):
    fig.text(-0.02, 1 - (i + 0.5) / len(PANEL_ELEMENTS), element,
             ha="left", va="center", fontsize=FONT["row_label"],
             rotation=90, fontweight="bold")

# Save figure
if SAVE_FIG:
    save_figure(fig, OUTPUT_NAME, formats=("png", "pdf"),
                dpi=300, transparent=False)
plt.show()